In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/creditcard.csv")

df.head()

In [ ]:
##getting dataset shape and exploring data
df.shape

In [ ]:
df.info()

In [ ]:
##so our target column is--->class
df.describe()

In [ ]:
#checking for imbalance
df['Class'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='Class',data=df)
plt.show()

CONCLUSION:-
So from this we can see that our data is imbalance.so we need to balance it

FEATURE UNDERSTANDING:-
| Feature | Meaning              |
| ------- | -------------------- |
| Time    | Transaction sequence |
| Amount  | Transaction value    |
| V1–V28  | PCA features         |
| Class   | Target               |



In [ ]:
df = pd.read_csv("../data/creditcard.csv")

df.head()

In [ ]:
#train-test split
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
y_train

In [ ]:
y_test

💥 IMPORTANT:

👉 stratify=y ensures:

Same fraud ratio in train & test

👉 VERY important for imbalanced data

STEP 3: FEATURE SCALING

Why?
Amount is not scaled
Model performs better with scaling

In [ ]:
#feature scaling
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler();

X_train['Amount']=scaler.fit_transform(X_train[['Amount']])
X_test['Amount']=scaler.transform(X_test[['Amount']])

TEP 4: HANDLE IMBALANCE (CRITICAL STEP)

⚠️ Problem:

Fraud = very rare

🔥 Solution: SMOTE (Oversampling)

In [ ]:
from imblearn.over_sampling import SMOTE

smote=SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
X_train.shape

In [ ]:
X_train_res.shape

In [ ]:
y_train.value_counts()

In [ ]:
y_train_res.value_counts()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
lr = LogisticRegression(max_iter=5000)

lr.fit(X_train_res, y_train_res)

y_pred_lr = lr.predict(X_test)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train_res, y_train_res)

y_pred_rf = rf.predict(X_test)

In [ ]:
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='logloss'
)

xgb.fit(X_train_res, y_train_res)

y_pred_xgb = xgb.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print("Logistic Regression:\n", classification_report(y_test, y_pred_lr))

print("Random Forest:\n", classification_report(y_test, y_pred_rf))

print("XGBoost:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.metrics import confusion_matrix

print("Logistic Regression:\n",confusion_matrix(y_test, y_pred_lr))
print("Random Forest:\n",confusion_matrix(y_test, y_pred_rf))
print("XGBoost:\n",confusion_matrix(y_test, y_pred_xgb))


1.LOGISTIC REGRESSION

Class 1:

Precision = 0.13 😬

Recall = 0.90 🔥

F1 = 0.23 ❌


🧠 Interpretation:

👉 Model is:

Catching fraud (high recall) ✔️

But making MANY wrong fraud predictions ❌

💥 Meaning:
👉 It says “fraud” too often

👉 Many false alarms

❌ Problem:

Precision very low

F1 very bad



🌳 2. RANDOM FOREST

Class 1:

Precision = 0.83 ✔️

Recall = 0.83 ✔️

F1 = 0.83 🔥

🧠 Interpretation:

👉 Very balanced model:

Good fraud detection ✔️

Few false alarms ✔️

💥 Meaning:

👉 Reliable model

👉 Strong performance



⚡ 3. XGBOOST

Class 1:

Precision = 0.36 😬

Recall = 0.89 🔥

F1 = 0.51 ⚠️

🧠 Interpretation:

👉 Similar to Logistic Regression but better:

High recall ✔️

Low precision ❌


💥 Meaning:

👉 Detects fraud well

👉 But still too many false positives


| Model         | Precision  | Recall  | F1        | Verdict      |
| ------------- | ---------- | ------- | --------- | ------------ |
| Logistic      | ❌ Very low | ✔️ High | ❌ Bad     | Not good     |
| Random Forest | ✔️ High    | ✔️ High | 🔥 Best   | ✅ Best       |
| XGBoost       | ❌ Low      | ✔️ High | ⚠️ Medium | Needs tuning |


In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Get reports as dictionary
lr_report = classification_report(y_test, y_pred_lr, output_dict=True)
rf_report = classification_report(y_test, y_pred_rf, output_dict=True)
xgb_report = classification_report(y_test, y_pred_xgb, output_dict=True)

# Create comparison table (focus on Class 1 = fraud)
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Precision": [
        lr_report['1']['precision'],
        rf_report['1']['precision'],
        xgb_report['1']['precision']
    ],
    "Recall": [
        lr_report['1']['recall'],
        rf_report['1']['recall'],
        xgb_report['1']['recall']
    ],
    "F1-Score": [
        lr_report['1']['f1-score'],
        rf_report['1']['f1-score'],
        xgb_report['1']['f1-score']
    ]
})

# Display table
comparison

In [ ]:
comparison.sort_values(by="F1-Score", ascending=False)

FEATURE IMPORTANCE (XGBOOST)

In [ ]:
feature_names = X_train.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb.feature_importances_
})

# Sort by importance
importance_df = importance_df.sort_values(by="Importance", ascending=False)

# Show top 10
importance_df.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"][:10], importance_df["Importance"][:10])
plt.gca().invert_yaxis()
plt.title("Top 10 Important Features (XGBoost)")
plt.xlabel("Importance")
plt.show()

In [ ]:
##SAVING BEST MODEL
best_model = rf

In [ ]:
import os
import joblib

# get project root
base_dir = os.path.abspath("..")

artifacts_path = os.path.join(base_dir, "artifacts")

os.makedirs(artifacts_path, exist_ok=True)

model_path = os.path.join(artifacts_path, "random_forest.pkl")

joblib.dump(rf, model_path)

In [ ]:
joblib.dump(scaler, "../artifacts/scaler.pkl")

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1]
}

xgb = XGBClassifier(eval_metric='logloss')

grid = GridSearchCV(
    xgb,
    param_grid,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train_res, y_train_res)

best_xgb = grid.best_estimator_

In [ ]:
from sklearn.metrics import classification_report

y_pred_xgb_tuned = best_xgb.predict(X_test)

print(classification_report(y_test, y_pred_xgb_tuned))

In [ ]:
joblib.dump(best_xgb, "../artifacts/xgb_tuned.pkl")

In [ ]:
##test pipeline
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.pipeline.predict_pipeline import PredictPipeline
import pandas as pd

# Load data
df = pd.read_csv("../data/creditcard.csv")
X = df.drop("Class", axis=1)

# Sample
sample = X.iloc[[0]]

pipeline = PredictPipeline()

pred, prob = pipeline.predict(sample)

print("Prediction:", pred)
print("Probability:", prob)

In [ ]:
import pandas as pd

sample = pd.DataFrame([{
    "Time": 50000,
    "V1": -1.359807,
    "V2": -0.072781,
    "V3": 2.536347,
    "V4": 1.378155,
    "V5": -0.338321,
    "V6": 0.462388,
    "V7": 0.239599,
    "V8": 0.098698,
    "V9": 0.363787,
    "V10": 0.090794,
    "V11": -0.551600,
    "V12": -0.617801,
    "V13": -0.991390,
    "V14": -0.311169,
    "V15": 1.468177,
    "V16": -0.470401,
    "V17": 0.207971,
    "V18": 0.025791,
    "V19": 0.403993,
    "V20": 0.251412,
    "V21": -0.018307,
    "V22": 0.277838,
    "V23": -0.110474,
    "V24": 0.066928,
    "V25": 0.128539,
    "V26": -0.189115,
    "V27": 0.133558,
    "V28": -0.021053,
    "Amount": 149.62
}])

In [ ]:
import sys
import os

# Fix path
sys.path.append(os.path.abspath(".."))

# Clear cache (optional)
sys.modules.pop("src.explanability.shap_explainer", None)

# Import
from src.explanability.shap_explainer import ShapExplainer

# Run
explainer = ShapExplainer()
shap_values = explainer.explain(sample)

print(shap_values)

In [ ]:
import shap

shap_values_raw = explainer.explainer.shap_values(sample)

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values_raw[0][:, 1],   # ✅ FIX HERE
        base_values=explainer.explainer.expected_value[1],
        data=sample.values[0],
        feature_names=sample.columns
    )
)

In [5]:
import requests

sample = {
  "Time": 90000,
  "V1": -10,
  "V2": 10,
  "V3": -8,
  "V4": 10,
  "V5": -7,
  "V6": -6,
  "V7": -9,
  "V8": 5,
  "V9": -6,
  "V10": -8,
  "V11": 6,
  "V12": -7,
  "V13": 2,
  "V14": -10,
  "V15": 1,
  "V16": -6,
  "V17": -9,
  "V18": -4,
  "V19": 2,
  "V20": 3,
  "V21": 4,
  "V22": 2,
  "V23": -2,
  "V24": -3,
  "V25": 1,
  "V26": -2,
  "V27": 3,
  "V28": 1,
  "Amount": 10000
}  # paste any above JSON

for _ in range(50):
    requests.post("http://127.0.0.1:8000/predict", json=sample)

In [ ]:
import sys
import os
import pandas as pd

# ✅ FIX PATH (NOTEBOOK VERSION)
sys.path.append(os.path.abspath(".."))

from src.monitoring.db import load_data_from_db

df = load_data_from_db()

print(df.shape)
df.head()

In [ ]:
import sys
import os
import pandas as pd

# ✅ FIX PATH (NOTEBOOK VERSION)
sys.path.append(os.path.abspath(".."))

from src.monitoring.drift import detect_drift

detect_drift("../data/creditcard.csv")

In [6]:
import requests

safe_sample = {
  "Time": 10000,
  "V1": 0.1,
  "V2": -0.2,
  "V3": 0.3,
  "V4": 0.1,
  "V5": -0.1,
  "V6": 0.2,
  "V7": 0.1,
  "V8": -0.1,
  "V9": 0.05,
  "V10": -0.05,
  "V11": 0.1,
  "V12": -0.1,
  "V13": 0.05,
  "V14": -0.05,
  "V15": 0.02,
  "V16": 0.01,
  "V17": -0.02,
  "V18": 0.01,
  "V19": 0.03,
  "V20": 0.02,
  "V21": -0.01,
  "V22": 0.01,
  "V23": 0.0,
  "V24": 0.02,
  "V25": 0.01,
  "V26": -0.01,
  "V27": 0.0,
  "V28": 0.0,
  "Amount": 120
}   # paste safe JSON
fraud_sample = {
  "Time": 75000,
  "V1": -6.5,
  "V2": 5.2,
  "V3": -4.8,
  "V4": 7.5,
  "V5": -3.0,
  "V6": -2.5,
  "V7": -4.5,
  "V8": 2.0,
  "V9": -3.5,
  "V10": -5.0,
  "V11": 4.0,
  "V12": -3.5,
  "V13": 1.0,
  "V14": -7.0,
  "V15": 0.5,
  "V16": -3.0,
  "V17": -5.5,
  "V18": -2.0,
  "V19": 1.5,
  "V20": 1.0,
  "V21": 2.0,
  "V22": 0.5,
  "V23": -0.5,
  "V24": -1.0,
  "V25": 0.2,
  "V26": -0.5,
  "V27": 0.8,
  "V28": 0.3,
  "Amount": 3000
}  # paste fraud JSON

# Insert safe data
for _ in range(10):
    requests.post("http://127.0.0.1:8000/predict", json=safe_sample)

# Insert fraud data
for _ in range(80):
    requests.post("http://127.0.0.1:8000/predict", json=fraud_sample)

In [7]:
import sys
import os
import pandas as pd

# ✅ FIX PATH (NOTEBOOK VERSION)
sys.path.append(os.path.abspath(".."))

from src.monitoring.db import load_data_from_db

df = load_data_from_db()
print(df["prediction"].value_counts())

prediction
1    230
0     60
Name: count, dtype: int64


In [ ]:
import sys
import os
import pandas as pd

# ✅ FIX PATH (NOTEBOOK VERSION)
sys.path.append(os.path.abspath(".."))

from src.monitoring.db import load_data_from_db

from src.pipeline.retrain_pipeline import retrain

retrain()

In [10]:
import requests

sample={
  "Time": 90000,
  "V1": -4.2,
  "V2": 3.8,
  "V3": -3.5,
  "V4": 5.5,
  "V5": -2.8,
  "V6": -2.2,
  "V7": -3.9,
  "V8": 1.8,
  "V9": -2.7,
  "V10": -4.1,
  "V11": 3.2,
  "V12": -2.9,
  "V13": 0.9,
  "V14": -5.8,
  "V15": 0.6,
  "V16": -2.5,
  "V17": -4.3,
  "V18": -1.9,
  "V19": 1.2,
  "V20": 0.9,
  "V21": 1.8,
  "V22": 0.4,
  "V23": -0.6,
  "V24": -0.9,
  "V25": 0.3,
  "V26": -0.4,
  "V27": 0.7,
  "V28": 0.25,
  "Amount": 2500
}

for _ in range(350):
    requests.post("http://127.0.0.1:8000/predict", json=sample)

In [11]:
sample={
  "Time": 52000,
  "V1": 0.8,
  "V2": -0.9,
  "V3": 0.95,
  "V4": 0.7,
  "V5": -0.75,
  "V6": 0.85,
  "V7": 0.9,
  "V8": -0.8,
  "V9": 0.7,
  "V10": -0.65,
  "V11": 0.88,
  "V12": -0.92,
  "V13": 0.05,
  "V14": -0.04,
  "V15": 0.02,
  "V16": 0.01,
  "V17": -0.02,
  "V18": 0.01,
  "V19": 0.03,
  "V20": 0.02,
  "V21": -0.01,
  "V22": 0.02,
  "V23": 0.00,
  "V24": 0.02,
  "V25": 0.01,
  "V26": -0.01,
  "V27": 0.00,
  "V28": 0.00,
  "Amount": 180
}

for _ in range(350):
    requests.post("http://127.0.0.1:8000/predict", json=sample)